# HAVEN — Agent & Scenario Simulator

Orchestrates all components into a single conversational agent:

- Risk signals (weather, regional, health) feed as context
- Inventory gaps feed as personalised kit state
- RAG retriever provides cited EU guidance
- Intent routing, tool dispatch, scenario simulation

Architecture: **structured router** (not ReAct loop)
The user query is classified into one of 5 intents, tools are called
deterministically, and the LLM composes the final cited answer.

```
User query
    ↓
Intent classifier (keyword + LLM fallback)
    ↓
Deterministic tool dispatch
    ├── get_risk_score       → current HavenSignals
    ├── get_kit_gaps         → InventoryReport
    ├── retrieve_guidelines  → top-k RAG chunks
    └── run_scenario         → survival estimate
    ↓
Answer composer (LLM)
    ↓
Cited answer + fallback if uncertain
```

## 0. Setup

In [1]:
%pip install langgraph langchain-core --verbose

Using pip 26.0.1 from /home/ildebrando/.pyenv/versions/3.12.9/envs/prepsense/lib/python3.12/site-packages/pip (python 3.12)
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Link requires a different Python (3.12.9 not in: '<3.12,>=3.9.0'): https://pypi.tuna.tsinghua.edu.cn/packages/5c/79/de8c27a3537cd794a15550def5268286890b7c74d191914b54373a9395da/langgraph_sdk-0.1.0-py3-none-any.whl (from https://pypi.tuna.tsinghua.edu.cn/simple/langgraph-sdk/) (requires-python:<3.12,>=3.9.0)
  Link requires a different Python (3.12.9 not in: '<3.12,>=3.9.0'): https://pypi.tuna.tsinghua.edu.cn/packages/5d/6e/216d56ac7066e0fefa3b12881aa92bd243d7cc86cb34a52892abaea47ebe/langgraph_sdk-0.1.0.tar.gz (from https://pypi.tuna.tsinghua.edu.cn/simple/langgraph-sdk/) (requires-python:<3.12,>=3.9.0)
  Link requires a different Python (3.12.9 not in: '<3.12,>=3.9.0'): https://pypi.tuna.tsinghua.edu.cn/packages/a0/76/d22975788ffc5f9d30dbe8e8cb11c10e0dfb8c6d182076cdf75eadd4ef21/langgraph_sdk-0.1.1-py3-

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path(".")
FAISS_DIR    = PROJECT_ROOT / "data" / "faiss"

print(f"Project root : {PROJECT_ROOT.resolve()}")
print(f"GROQ_API_KEY : {'set' if os.getenv('GROQ_API_KEY') else 'NOT SET'}")
print(f"LLM_BACKEND  : {os.getenv('LLM_BACKEND', 'ollama (default)')}") 

Project root : /home/ildebrando/code/ijesusjr/000_DS_Portfolio/02_prepsense
GROQ_API_KEY : set
LLM_BACKEND  : groq


---
## 1. Load Components

Loads the RAG retriever, LLM backend, inventory report, and risk signals.
Run this after the RAG notebook has built the FAISS index.

In [ ]:
# RAG retriever
from rag.retriever import HavenRetriever

retriever = HavenRetriever.from_disk(
    index_path= str(FAISS_DIR / "index.bin"),
    meta_path=  str(FAISS_DIR / "chunks.json"),
)
print(f"Retriever    : {retriever.index.ntotal} chunks")

In [ ]:
# LLM backend
from rag.llm import HavenLLM

BACKEND = os.getenv("LLM_BACKEND", None)
if BACKEND is None:
    for b in ["groq", "ollama", "anthropic"]:
        if HavenLLM(backend=b).is_available():
            BACKEND = b
            break

llm = HavenLLM(backend=BACKEND) if BACKEND else None
print(f"LLM backend  : {BACKEND or 'not configured'}")
print(f"LLM available: {llm.is_available() if llm else False}")

In [ ]:
# Inventory analysis
from core.inventory_analyzer import KitItem, analyze_inventory
from datetime import date

# Realistic partially-stocked kit for 1 person
sample_kit = [
    KitItem(name="Drinking water",        category="water",   quantity=2.0,  eu_recommended=9.0,  unit="liters", expiry_date=None),
    KitItem(name="Non-perishable food",   category="food",    quantity=1.0,  eu_recommended=3.0,  unit="days",   expiry_date=None),
    KitItem(name="First aid kit",         category="meds",    quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Regular medication",    category="meds",    quantity=0.0,  eu_recommended=7.0,  unit="days",   expiry_date=None),
    KitItem(name="Battery-powered radio", category="comms",   quantity=0.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Flashlight",            category="light",   quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Cash",                  category="cash",    quantity=20.0, eu_recommended=70.0, unit="EUR",    expiry_date=None),
    KitItem(name="Hand sanitizer",        category="hygiene", quantity=0.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
]
inv_report = analyze_inventory(sample_kit)
print(f"Inventory    : {len(inv_report.gaps)} gaps, score {inv_report.total_gap_score}/100")

In [ ]:
# Risk signals (simulated if live fetch not available)
from core.risk_engine import RiskResult, HavenSignals

# Simulate realistic signals for Barcelona in April
risk_result = RiskResult(
    risk_score=25, risk_level="ELEVATED",
    weather_severity=0, alert_severity=25,
    wind_bonus=0, rain_bonus=0,
)
signals = HavenSignals(
    weather=            risk_result,
    geo_score=          4,
    geo_trend=          "STABLE",
    geo_country=        "Spain",
    health_score=       19,
    health_level=       "ELEVATED",
    top_health_threats= ["Avian Influenza", "Dengue"],
)
print(f"Risk signals : weather={signals.weather.risk_score}/100 | geo={signals.geo_score}/30 | health={signals.health_score}/50")

---
## 2. Agent Tools

Four tools — each a pure function, no LLM dependency.
The agent calls them deterministically based on query intent.

In [6]:
from agent.tools import get_risk_score, get_kit_gaps, retrieve_guidelines, run_scenario

# Tool 1: get_risk_score
print("=== Tool: get_risk_score ===")
risk_summary = get_risk_score(signals)
print(risk_summary.to_prompt_str())
print(f"\nOverall concern: {risk_summary.overall_concern}")
print(f"Narrative      : {risk_summary.narrative}")

=== Tool: get_risk_score ===
CURRENT RISK SIGNALS:
  Weather  : 25/100 — ELEVATED (alert active, severity score 25)
  Regional : 4/30  — LOW (trend: STABLE)
  Health   : 19/50  — ELEVATED | Active: Avian Influenza, Dengue
  Overall  : MODERATE

Overall concern: MODERATE
Narrative      : Current concern: health ELEVATED.


In [7]:
# Tool 2: get_kit_gaps
print("=== Tool: get_kit_gaps ===")
gap_summary = get_kit_gaps(inv_report)
print(gap_summary.to_prompt_str())

=== Tool: get_kit_gaps ===
KIT GAPS (6 items, score 36/100):
  [HIGH  ] Regular medication        0.0/7.0 days (100% missing)
  [HIGH  ] Drinking water            2.0/9.0 liters (78% missing)
  [HIGH  ] Non-perishable food       1.0/3.0 days (67% missing)
  [MEDIUM] Battery-powered radio     0.0/1.0 units (100% missing)
  [LOW   ] Hand sanitizer            0.0/1.0 units (100% missing)
  [LOW   ] Cash                      20.0/70.0 EUR (71% missing)


In [8]:
# Tool 3: retrieve_guidelines
print("=== Tool: retrieve_guidelines ===")
gl = retrieve_guidelines(retriever, "why do I need water in my emergency kit?", k=3)
print(f"Query   : {gl.query}")
print(f"Chunks  : {len(gl.chunks)}")
for i, c in enumerate(gl.chunks):
    label = c.source.split("(")[0].strip()
    print(f"  [{c.score:.3f}] {label}, p{c.page}: {c.text[:80]}...")

=== Tool: retrieve_guidelines ===


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3333.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query   : why do I need water in my emergency kit?
Chunks  : 3
  [0.514] Belgium Emergency Kit Guide, p2: Acollar / harness and a leash. Near your backpack, a carrier adapted to your pet...
  [0.493] Belgium Emergency Kit Guide, p1: National Crisis Center AN EMERGENCY KIT AT HOME An emergency kit is a set of ite...
  [0.487] Belgium Emergency Kit Guide, p2: these on the recommendation of the authorities. To find out more, visit: Light, ...


In [ ]:
# Tool 4: run_scenario — the HAVEN "wow" feature
print("=== Tool: run_scenario ===\n")

scenarios = [
    ("power_outage", 72,  1),
    ("power_outage", 72,  2),
    ("flood",        48,  1),
    ("heat_wave",    96,  2),
    ("earthquake",   72,  1),
]

for event, hours, people in scenarios:
    result = run_scenario(inv_report, event_type=event, duration_hours=hours, people=people)
    print(f"{event:<15} {hours}h {people}p → {result.survival_pct:>3}% covered")
    print(f"  Water: {result.water_hours:.0f}h | Food: {result.food_hours:.0f}h | "
          f"Comms: {'OK' if result.comms_ok else 'NONE'} | Meds: {'OK' if result.meds_ok else 'NONE'}")
    if result.critical_gaps:
        print(f"  ⚠ First failure: {result.critical_gaps[0]}")
    print()

In [10]:
# Scenario detail — full output for power outage
print("=== Scenario Detail: 72h Power Outage (2 people) ===\n")
detail = run_scenario(inv_report, event_type="power_outage", duration_hours=72, people=2)
print(detail.to_prompt_str())
print("\nRecommendations:")
for r in detail.recommendations:
    print(f"  • {r}")

=== Scenario Detail: 72h Power Outage (2 people) ===

SCENARIO ANALYSIS: Power Outage (72h, 2 person(s))
  Water supply    : lasts 8h (INSUFFICIENT)
  Food supply     : lasts 12h (INSUFFICIENT)
  Lighting        : NONE
  Communications  : NONE — cannot receive alerts
  Medication      : MISSING
  Coverage        : 19% of 72h scenario
  Critical gaps   : Water runs out at hour 8 (need 18.0L, have 2.0L), Food runs out at hour 12, No communications device — cannot receive emergency alerts, No medication stock
  Summary: Your kit covers approximately 19% of a 72h power outage for 2 person(s). First critical shortage at hour 8.

Recommendations:
  • Store 16.0L more water (3.0L/person/day × 2 people × 3 days)
  • Add 5.0 more days of food per person
  • Get a battery-powered radio or charged power bank
  • Keep a 7-day supply of regular medication
  • Add flashlight with spare batteries or candles


---
## 3. Intent Router

Classifies user queries into one of 5 intents using keyword matching
(with optional LLM fallback for ambiguous queries).

Why keyword routing instead of pure LLM routing?
- Deterministic — same query always routes the same way
- Zero latency — no LLM call needed for routing
- Easy to debug — routing decisions are explicit
- LLM is reserved for what it's best at: composing the answer

In [29]:
from agent.router import route

test_queries = [
    "Why do I need a battery-powered radio?",
    "How dangerous is my current situation?",
    "How long will my kit last in a 72h power outage?",
    "How should I prepare an evacuation plan?",
    "What food should I store for emergencies?",
    "Is my current risk level high?",
    "I need medication, what should I have?",
    "What if there is a flood for 48 hours?",
    "How do I prepare my family for a crisis?",
    "What is the best way to survive a blackout?",
]

print(f"{'Query':<52} {'Intent':<16} {'Confidence'}")
print("-" * 80)
for q in test_queries:
    d = route(q)
    print(f"{q[:50]:<52} {d.intent:<16} {d.confidence}")

Query                                                Intent           Confidence
--------------------------------------------------------------------------------
Why do I need a battery-powered radio?               KIT_QUESTION     HIGH
How dangerous is my current situation?               RISK_QUESTION    HIGH
How long will my kit last in a 72h power outage?     SCENARIO         HIGH
How should I prepare an evacuation plan?             GENERAL_PREP     HIGH
What food should I store for emergencies?            KIT_QUESTION     MEDIUM
Is my current risk level high?                       RISK_QUESTION    HIGH
I need medication, what should I have?               KIT_QUESTION     HIGH
What if there is a flood for 48 hours?               SCENARIO         HIGH
How do I prepare my family for a crisis?             GENERAL_PREP     MEDIUM
What is the best way to survive a blackout?          SCENARIO         HIGH


---
## 4. LangGraph Agent

`HavenAgent` wraps the LangGraph graph into a simple `.ask()` interface.

The graph has 6 nodes:
```
route → risk → gaps → scenario → guidelines → compose → END
```
Each tool node checks whether it's needed for the current intent before running.
The LLM is called once at the `compose` node to generate the final answer.

In [ ]:
from agent.agent import HavenAgent

agent = HavenAgent(
    retriever=   retriever,
    inv_report=  inv_report,
    signals=     signals,
    llm=         llm,
    people=      1,
)
print("Agent ready")
print(f"  LangGraph nodes : route → risk → gaps → scenario → guidelines → compose")
print(f"  LLM backend     : {BACKEND or 'not configured (rule-based answers)'}")

---
## 5. End-to-End Query Tests

10 realistic queries covering all intents and scenarios.
Each shows the routing decision, tools called, and final answer.

In [31]:
# Query 1/10
response = agent.ask(
    query="Why do I need a battery-powered radio?",
    event_type="general",
    duration_hours=72,
)
agent.print_response(response)


Q: Why do I need a battery-powered radio?
Intent  : KIT_QUESTION  |  Routing: [keyword] Keyword match: 3 hits for KIT_QUESTION.
Fallback: False

You need a battery-powered radio to stay informed during an emergency situation when power is out. According to the Belgium Emergency Kit Guide [Source: Belgium Emergency Kit Guide, Page 2], it's recommended to bring a battery-powered or wind-up radio to listen to the media.

Considering your current kit gaps, you are missing a battery-powered radio entirely, with a score of 0.0/1.0 units [HIGH  ] Battery-powered radio. I recommend prioritizing the purchase or acquisition of a battery-powered radio to address this critical gap.

Additionally, having a battery-powered radio will also be useful for staying informed about the situation, which is crucial for making informed decisions about your safety and well-being.

--- Sources ---
  • Belgium Emergency Kit Guide (crisiscenter.be), p.2
  • Sweden Emergency Guide (krisinformation.se), p.1
  • Cz

In [32]:
# Query 2/10
response = agent.ask(
    query="How long will my kit last in a 72-hour power outage?",
    event_type="power_outage",
    duration_hours=72,
)
agent.print_response(response)


Q: How long will my kit last in a 72-hour power outage?
Intent  : SCENARIO  |  Routing: [keyword] Keyword match: 5 hits for SCENARIO.
Fallback: False

Based on the provided guidelines and the user's current kit gaps, I can assess how long their kit will last in a 72-hour power outage.

According to the Belgium Emergency Kit Guide [Source: Belgium Emergency Kit Guide, Page 1], it is recommended to have an emergency kit at home that can sustain you for at least 3 days (72 hours).

The user's current kit gaps indicate that they have insufficient supplies for a 72-hour power outage. Specifically:

- Drinking water: 2.0/9.0 liters (78% missing) [Source: user's kit gaps]
- Non-perishable food: 1.0/3.0 days (67% missing) [Source: user's kit gaps]
- Medication: 0.0/7.0 days (100% missing) [Source: user's kit gaps]

Considering these gaps, the user's kit will likely not last the full 72 hours. The critical shortages will occur at:

- Hour 16: Water runs out (need 9.0L, have 2.0L) [Source: scen

In [33]:
# Query 3/10
response = agent.ask(
    query="What is my current risk level?",
    event_type="general",
    duration_hours=72,
)
agent.print_response(response)


Q: What is my current risk level?
Intent  : RISK_QUESTION  |  Routing: [keyword] Keyword match: 3 hits for RISK_QUESTION.
Fallback: False

Based on the provided information, your current overall risk level is MODERATE. This is indicated by the Overall risk score of 25/100, which is classified as MODERATE according to the EU guidelines.

However, I would like to highlight a few critical kit gaps that are relevant to your current risk level:

1. **Water storage**: With an elevated risk of Avian Influenza and Dengue, it's essential to have a sufficient water supply for at least 1 liter per person per day for 3-5 days [Source: EU Civil Protection and Humanitarian Aid Operations, "Emergency Preparedness and Response", Page 12]. Your current water storage capacity is not specified, so I recommend checking your inventory and aiming for at least 3-5 days' worth of water for 1 person.
2. **First aid kit**: With an elevated risk of Avian Influenza and Dengue, it's crucial to have a well-stocked

In [34]:
# Query 4/10
response = agent.ask(
    query="How should I store water safely?",
    event_type="general",
    duration_hours=72,
)
agent.print_response(response)


Q: How should I store water safely?
Intent  : KIT_QUESTION  |  Routing: [keyword] Keyword match: 2 hits for KIT_QUESTION.
Fallback: False

Answer for: How should I store water safely?


KIT GAPS (6 items, score 36/100):
  [HIGH  ] Regular medication        0.0/7.0 days (100% missing)
  [HIGH  ] Drinking water            2.0/9.0 liters (78% missing)
  [HIGH  ] Non-perishable food       1.0/3.0 days (67% missing)
  [MEDIUM] Battery-powered radio     0.0/1.0 units (100% missing)
  [LOW   ] Hand sanitizer            0.0/1.0 units (100% missing)
  [LOW   ] Cash                      20.0/70.0 EUR (71% missing)


From official EU guidelines:

[1] Source: Belgium Emergency Kit Guide (crisiscenter.be), Page 2 (score: 0.52)
Acollar / harness and a leash. Near your backpack, a carrier adapted to your pet. Do you have exotic pets? Make sure you can transport them safely, and that you include the necessary equipment (food, etc.) in your emergency kit. Things to have at home Food and drinks e Bottl

In [35]:
# Query 5/10
response = agent.ask(
    query="What if there is a flood for 48 hours?",
    event_type="flood",
    duration_hours=48,
)
agent.print_response(response)


Q: What if there is a flood for 48 hours?
Intent  : SCENARIO  |  Routing: [keyword] Keyword match: 3 hits for SCENARIO.
Fallback: False

Answer for: What if there is a flood for 48 hours?


SCENARIO ANALYSIS: Flood (48h, 1 person(s))
  Water supply    : lasts 12h (INSUFFICIENT)
  Food supply     : lasts 24h (INSUFFICIENT)
  Lighting        : NONE
  Communications  : NONE — cannot receive alerts
  Medication      : MISSING
  Coverage        : 31% of 48h scenario
  Critical gaps   : Water runs out at hour 12 (need 8.0L, have 2.0L), Food runs out at hour 24, No communications device — cannot receive emergency alerts, No medication stock
  Summary: Your kit covers approximately 31% of a 48h flood for 1 person(s). First critical shortage at hour 12.


Recommendations:

  • Store 6.0L more water (4.0L/person/day × 1 people × 2 days)

  • Add 1.0 more days of food per person

  • Get a battery-powered radio or charged power bank

  • Keep a 7-day supply of regular medication

KIT GAPS (6 ite

In [36]:
# Query 6/10
response = agent.ask(
    query="What food should I store for emergencies?",
    event_type="general",
    duration_hours=72,
)
agent.print_response(response)


Q: What food should I store for emergencies?
Intent  : KIT_QUESTION  |  Routing: [keyword] Keyword match: 2 hits for KIT_QUESTION.
Fallback: False

Answer for: What food should I store for emergencies?


KIT GAPS (6 items, score 36/100):
  [HIGH  ] Regular medication        0.0/7.0 days (100% missing)
  [HIGH  ] Drinking water            2.0/9.0 liters (78% missing)
  [HIGH  ] Non-perishable food       1.0/3.0 days (67% missing)
  [MEDIUM] Battery-powered radio     0.0/1.0 units (100% missing)
  [LOW   ] Hand sanitizer            0.0/1.0 units (100% missing)
  [LOW   ] Cash                      20.0/70.0 EUR (71% missing)


From official EU guidelines:

[1] Source: Netherlands Emergency Kit Guide (denkvooruit.nl), Page 1 (score: 0.63)
Store the items in a handy, easily accessible place. It is also handy to put a bag in this place, in case you suddenly have to leave your house. You can put things in there and easily take them with you. Think for example of your keys, ID and cash. Perso

In [37]:
# Query 7/10
response = agent.ask(
    query="I have no medication — what do the EU guides recommend?",
    event_type="general",
    duration_hours=72,
)
agent.print_response(response)


Q: I have no medication — what do the EU guides recommend?
Intent  : KIT_QUESTION  |  Routing: [keyword] Keyword match: 2 hits for KIT_QUESTION.
Fallback: False

Answer for: I have no medication — what do the EU guides recommend?


KIT GAPS (6 items, score 36/100):
  [HIGH  ] Regular medication        0.0/7.0 days (100% missing)
  [HIGH  ] Drinking water            2.0/9.0 liters (78% missing)
  [HIGH  ] Non-perishable food       1.0/3.0 days (67% missing)
  [MEDIUM] Battery-powered radio     0.0/1.0 units (100% missing)
  [LOW   ] Hand sanitizer            0.0/1.0 units (100% missing)
  [LOW   ] Cash                      20.0/70.0 EUR (71% missing)


From official EU guidelines:

[1] Source: Belgium Emergency Kit Guide (crisiscenter.be), Page 1 (score: 0.45)
GP, your insurance company). Copies of the identity cards of the members of your household. Apen and paper. A basic first-aid kit: bandages, dressings, compresses, scissors, tweezers, safety pins, painkillers, disinfectants, etc.

In [38]:
# Query 8/10
response = agent.ask(
    query="How do I prepare my family for a crisis?",
    event_type="general",
    duration_hours=72,
)
agent.print_response(response)


Q: How do I prepare my family for a crisis?
Intent  : GENERAL_PREP  |  Routing: [keyword] Keyword match: 2 hits for GENERAL_PREP.
Fallback: False

Answer for: How do I prepare my family for a crisis?



From official EU guidelines:

[1] Source: Sweden Emergency Guide (krisinformation.se), Page 1 (score: 0.77)
stop working. List, on paper, of phone numbers for family, relatives, neighbours, hospital, municipality, emergency services, energy provider. HOME PREPAREDNESS QUESTIONS AND ANSWERS What do you want me to do? - It's good if you: Familiarise yourself with various types of crises - and how they can affect you and your family. + Have at least a week's worth of home preparedness supplies, enabling you to function during a crisis or war. Stay alert when it comes to the spread of fake news and rumours. I can’t store a wee...


[Rule-based answer — LLM not available]

[LLM error: 429 Client Error: Too Many Requests for url: https://api.groq.com/openai/v1/chat/completions]

--- Sources 

In [ ]:
# Query 9/10 — household of 2 people, 96h heat wave
# Pass people=2 explicitly so the scenario overrides the agent's default (1 person)
response = agent.ask(
    query="What happens to my kit in a 96h heat wave for 2 people?",
    event_type="heat_wave",
    duration_hours=96,
    people=2,
)
agent.print_response(response)

In [41]:
# Query 10/10
response = agent.ask(
    query="What is a good evacuation plan?",
    event_type="general",
    duration_hours=72,
)
agent.print_response(response)


Q: What is a good evacuation plan?
Intent  : GENERAL_PREP  |  Routing: [keyword] Keyword match: 2 hits for GENERAL_PREP.
Fallback: False

Based on the provided EU emergency preparedness guidelines, I'll provide a practical answer to the user's question about a good evacuation plan.

According to the Belgium Emergency Kit Guide [Source: Belgium Emergency Kit Guide (crisiscenter.be), Page 2], it's essential to have an evacuation plan in place. The guide suggests discussing with your neighbors how you can help each other in case of an emergency. This is crucial, especially if you live in a block of flats, a student room, or other forms of co-habitation.

To create a good evacuation plan, consider the following steps:

1. Identify the safest route to exit your home, taking into account any potential hazards such as fires or floods.
2. Designate a meeting point outside your home where everyone can gather once they've evacuated.
3. Make sure all household members know the evacuation plan an

---
## 6. Fallback Behaviour

When the agent cannot answer from available sources — either because
the intent is UNKNOWN or no relevant guidelines are retrieved — it
surfaces official source links rather than guessing.

This is a deliberate design choice for a safety-critical domain:
better to say "I don't know, check here" than to hallucinate.

In [42]:
# Test UNKNOWN intent → fallback response
fallback_queries = [
    "What is the weather like in Tokyo?",
    "How do I cook pasta?",
    "What is the meaning of life?",
]

for q in fallback_queries:
    r = agent.ask(q)
    print(f"Q: {q}")
    print(f"  Intent  : {r.intent}")
    print(f"  Fallback: {r.fallback}")
    print(f"  Answer  : {r.answer[:120]}...")
    print()

Q: What is the weather like in Tokyo?
  Intent  : UNKNOWN
  Fallback: True
  Answer  : I wasn't able to find specific guidance for this question in the
official EU emergency preparedness documents.

For auth...

Q: How do I cook pasta?
  Intent  : UNKNOWN
  Fallback: True
  Answer  : I wasn't able to find specific guidance for this question in the
official EU emergency preparedness documents.

For auth...

Q: What is the meaning of life?
  Intent  : UNKNOWN
  Fallback: True
  Answer  : I wasn't able to find specific guidance for this question in the
official EU emergency preparedness documents.

For auth...



---
## 7. Scenario Simulator Deep Dive

The scenario simulator is the HAVEN "wow feature" — it gives a
personalised survival estimate based on the user's *actual* kit quantities.

This is what separates HAVEN from a generic chatbot:
- Generic: "You need 3L of water per person per day"
- HAVEN: "You have 2L. For 2 people in a 72h power outage you need
  18L. Your water runs out at hour 8. Here is what to buy."

In [43]:
# Show survival coverage across different household sizes and event durations
from agent.tools import run_scenario

print("Survival coverage by household size and event duration\n")
print(f"{'Event':<15} {'Hours':>6} {'People':>7} {'Coverage':>9} {'Water lasts':>12} {'Food lasts':>11}")
print("-" * 65)

for event in ["power_outage", "flood", "heat_wave", "earthquake"]:
    for hours in [24, 48, 72]:
        for people in [1, 2, 4]:
            r = run_scenario(inv_report, event_type=event, duration_hours=hours, people=people)
            print(f"{event:<15} {hours:>6} {people:>7} {r.survival_pct:>8}% "
                  f"{r.water_hours:>10.0f}h {r.food_hours:>10.0f}h")
    print()

Survival coverage by household size and event duration

Event            Hours  People  Coverage  Water lasts  Food lasts
-----------------------------------------------------------------
power_outage        24       1       54%         16h         24h
power_outage        24       2       33%          8h         12h
power_outage        24       4       22%          4h          6h
power_outage        48       1       33%         16h         24h
power_outage        48       2       22%          8h         12h
power_outage        48       4       17%          4h          6h
power_outage        72       1       26%         16h         24h
power_outage        72       2       19%          8h         12h
power_outage        72       4       15%          4h          6h

flood               24       1       50%         12h         24h
flood               24       2       31%          6h         12h
flood               24       4       21%          3h          6h
flood               48       1 

In [44]:
# Show what a well-stocked kit looks like in comparison
from core.inventory_analyzer import KitItem, analyze_inventory

full_kit = [
    KitItem(name="Drinking water",        category="water",   quantity=9.0,  eu_recommended=9.0,  unit="liters", expiry_date=None),
    KitItem(name="Non-perishable food",   category="food",    quantity=3.0,  eu_recommended=3.0,  unit="days",   expiry_date=None),
    KitItem(name="First aid kit",         category="meds",    quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Regular medication",    category="meds",    quantity=7.0,  eu_recommended=7.0,  unit="days",   expiry_date=None),
    KitItem(name="Battery-powered radio", category="comms",   quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Flashlight",            category="light",   quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
    KitItem(name="Cash",                  category="cash",    quantity=70.0, eu_recommended=70.0, unit="EUR",    expiry_date=None),
    KitItem(name="Hand sanitizer",        category="hygiene", quantity=1.0,  eu_recommended=1.0,  unit="units",  expiry_date=None),
]
full_inv = analyze_inventory(full_kit)

print("Comparison: partial kit vs full kit (72h power outage, 1 person)\n")
for label, report in [("Partial kit (current)", inv_report), ("Full kit (EU standard)", full_inv)]:
    r = run_scenario(report, event_type="power_outage", duration_hours=72, people=1)
    print(f"{label}:")
    print(f"  Survival coverage : {r.survival_pct}%")
    print(f"  Water duration    : {r.water_hours:.0f}h")
    print(f"  Food duration     : {r.food_hours:.0f}h")
    print(f"  Communications    : {'OK' if r.comms_ok else 'NONE'}")
    print(f"  Medication        : {'OK' if r.meds_ok else 'NONE'}")
    if r.critical_gaps:
        print(f"  Critical gaps     : {len(r.critical_gaps)}")
    else:
        print(f"  Critical gaps     : NONE — fully prepared")
    print()

Comparison: partial kit vs full kit (72h power outage, 1 person)

Partial kit (current):
  Survival coverage : 26%
  Water duration    : 16h
  Food duration     : 24h
  Communications    : NONE
  Medication        : NONE
  Critical gaps     : 4

Full kit (EU standard):
  Survival coverage : 12%
  Water duration    : 0h
  Food duration     : 0h
  Communications    : NONE
  Medication        : NONE
  Critical gaps     : 4



---
## 8. LLM Trade-off: Local vs API

This section documents the local↔cloud trade-off explicitly
for the README and portfolio documentation.

| Dimension | Ollama/Mistral (local) | Groq/Llama (deployed) |
|-----------|------------------------|----------------------|
| Cost | Free | Free (14,400 req/day) |
| Speed | 2-8s (GPU dependent) | ~0.5s (custom hardware) |
| Privacy | Fully local | Data sent to Groq |
| GPU required | Yes (8GB+ VRAM) | No |
| Offline capable | Yes | No |
| Quality | Good | Good |
| Deployment | Cannot (no GPU on free tier) | Yes |
| Switch cost | Change LLM_BACKEND env var | Change LLM_BACKEND env var |

In [ ]:
# Demonstrate the backend switch — identical interface, different backend
print("Trade-off demonstration\n")
print("Both backends use identical prompt template and tool results.")
print("Switch is one env var: LLM_BACKEND=groq or LLM_BACKEND=ollama")
print()

for backend_name in ["groq", "ollama", "anthropic"]:
    test_llm = HavenLLM(backend=backend_name)
    available = test_llm.is_available()
    latency   = "<1s" if backend_name == "groq" else "2-8s" if backend_name == "ollama" else "<2s"
    cost      = "Free (14,400/day)" if backend_name == "groq" else "Free (local GPU)" if backend_name == "ollama" else "~$0.25/1M tokens"
    gpu       = "No" if backend_name in ("groq", "anthropic") else "Yes (8GB+)"
    print(f"  {backend_name:<12} available={str(available):<6} latency={latency:<8} cost={cost:<22} gpu={gpu}")

---
## 9. Summary

In [ ]:
print("HAVEN Agent — Complete")
print("=" * 55)
print()
print("Agent components:")
print("  [x] get_risk_score       — HavenSignals → RiskSummary")
print("  [x] get_kit_gaps         — InventoryReport → GapSummary")
print("  [x] retrieve_guidelines  — query → RAG chunks")
print("  [x] run_scenario         — kit + event + duration → survival estimate")
print()
print("  [x] Intent router        — 5 intents, keyword + LLM fallback")
print("  [x] LangGraph graph      — 6 nodes, deterministic dispatch")
print("  [x] Answer composer      — LLM with all tool context")
print("  [x] Fallback handling    — UNKNOWN intent → official sources")
print("  [x] LLM trade-off        — documented local vs API")
print()
print("10 end-to-end queries tested across all intents:")
print("  KIT_QUESTION, RISK_QUESTION, SCENARIO, GENERAL_PREP, UNKNOWN")